# 1_EDA_e_Storytelling

Projeto Datathon Fase 5 - Associacao Passos Magicos.

Este notebook cobre:
- limpeza e integracao das abas de 2022, 2023 e 2024;
- padronizacao de inconsistencias comuns em dados educacionais;
- storytelling gerencial com graficos interativos (Plotly);
- resposta estruturada para as 11 perguntas do documento oficial;
- analise de matriz de correlacao global e por ano;
- insumos para o modelo preditivo de risco de defasagem.


## 1) Setup

In [1]:
import re
import unicodedata
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
px.defaults.template = "plotly_white"


def detect_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "app.py").exists():
            return candidate
    return current


PROJECT_ROOT = detect_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CONSOLIDATED_FALLBACK_PATH = PROCESSED_DIR / "pede_consolidado_2022_2024.csv"

if not DATA_PATH.exists() and not CONSOLIDATED_FALLBACK_PATH.exists():
    raise FileNotFoundError(
        "Nenhuma fonte encontrada. Esperado: "
        f"{DATA_PATH.resolve()} ou {CONSOLIDATED_FALLBACK_PATH.resolve()}"
    )

print("Fonte Excel preferencial:", DATA_PATH.resolve())
print("Fallback CSV:", CONSOLIDATED_FALLBACK_PATH.resolve())


Arquivo de entrada: C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\data\BASE DE DADOS PEDE 2024 - DATATHON.xlsx


## 2) Funcoes de limpeza e integracao

In [2]:
def normalize_column_name(col_name: str) -> str:
    """Normaliza nome de coluna removendo acentos, espacos e simbolos."""
    normalized = unicodedata.normalize("NFKD", str(col_name)).encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", normalized).strip("_").lower()
    return normalized


def clean_empty_strings(series: pd.Series) -> pd.Series:
    """Converte strings vazias e marcadores textuais de nulo para NaN."""
    cleaned = series.astype(str).str.strip()
    return cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})


def coerce_numeric_series(series: pd.Series) -> pd.Series:
    """Converte texto numerico (virgula, percentual, etc.) para float."""
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan, "INCLUIR": np.nan, "incluir": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def normalize_age_series(series: pd.Series) -> pd.Series:
    """Normaliza idade aceitando formatos numericos e datas serializadas de planilha."""
    numeric_age = coerce_numeric_series(series)
    date_values = pd.to_datetime(series, errors="coerce")

    age_from_date = np.where(
        date_values.notna() & (date_values.dt.year == 1900) & (date_values.dt.month == 1),
        date_values.dt.day,
        np.nan,
    )

    result = pd.Series(numeric_age, index=series.index)
    mask = result.isna() & ~pd.isna(age_from_date)
    result.loc[mask] = age_from_date[mask]
    result = result.where(result.between(6, 30))
    return result.round()


def coalesce_columns(df: pd.DataFrame, new_col: str, candidates: list[str], numeric: bool = False) -> None:
    """
    Cria uma coluna consolidada com a primeira informacao nao nula encontrada
    entre as candidatas, na ordem informada.
    """
    result = None
    for col in candidates:
        if col not in df.columns:
            continue
        values = coerce_numeric_series(df[col]) if numeric else clean_empty_strings(df[col])
        result = values if result is None else result.where(result.notna(), values)
    if result is None:
        result = pd.Series(np.nan, index=df.index)
    df[new_col] = result


def standardize_gender(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_clean = str(value).strip().lower()
    mapping = {
        "menino": "Masculino",
        "masculino": "Masculino",
        "menina": "Feminino",
        "feminino": "Feminino",
    }
    return mapping.get(value_clean, str(value).strip())


def standardize_stone(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_norm = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value_norm = value_norm.strip().lower()
    mapping = {
        "quartzo": "Quartzo",
        "agata": "Agata",
        "ametista": "Ametista",
        "topazio": "Topazio",
        "incluir": np.nan,
    }
    return mapping.get(value_norm, str(value).strip())


def extract_phase_code(value: str) -> str:
    """
    Extrai ALFA ou FASE X.
    Em 2022 ha casos numericos simples (ex.: "7"), tambem tratados aqui.
    """
    if pd.isna(value):
        return np.nan
    raw = str(value).strip().upper()
    if raw in {"", "NAN", "NONE"}:
        return np.nan
    if "ALFA" in raw:
        return "ALFA"
    match = re.search(r"FASE\s*([0-9]+)", raw)
    if match:
        return f"FASE {int(match.group(1))}"
    if re.fullmatch(r"[0-9]+", raw):
        phase_num = int(raw)
        return "ALFA" if phase_num == 0 else f"FASE {phase_num}"
    return np.nan


def load_and_prepare_data(data_path: Path) -> pd.DataFrame:
    """Le o Excel multi-aba e devolve um unico DataFrame harmonizado."""
    xls = pd.ExcelFile(data_path)
    unified_frames = []

    for sheet in xls.sheet_names:
        year_match = re.search(r"20\d{2}", sheet)
        if not year_match:
            continue
        year = int(year_match.group())

        raw = xls.parse(sheet)
        raw = raw.dropna(how="all").copy()
        raw.columns = [normalize_column_name(c) for c in raw.columns]

        if "ra" not in raw.columns:
            continue
        raw["ra"] = clean_empty_strings(raw["ra"])
        raw = raw[~raw["ra"].str.upper().isin(["RA-1", "NAN"])]
        raw = raw[raw["ra"].notna()]
        raw["ano_referencia"] = year

        coalesce_columns(raw, "nome_aluno", ["nome_anonimizado", "nome"], numeric=False)
        coalesce_columns(raw, "idade", ["idade", "idade_22"], numeric=False)
        coalesce_columns(raw, "data_nascimento", ["data_de_nasc", "ano_nasc"], numeric=False)
        coalesce_columns(raw, "instituicao_ensino", ["instituicao_de_ensino", "escola"], numeric=False)
        coalesce_columns(raw, "fase_ideal", ["fase_ideal", "fase_ideal_"], numeric=False)
        coalesce_columns(raw, "defasagem", ["defasagem", "defas"], numeric=True)

        coalesce_columns(raw, "nota_matematica", ["mat", "matem"], numeric=True)
        coalesce_columns(raw, "nota_portugues", ["por", "portug"], numeric=True)
        coalesce_columns(raw, "nota_ingles", ["ing", "ingles"], numeric=True)

        coalesce_columns(raw, "inde_ano", ["inde_2024", "inde_2023", "inde_23", "inde_22"], numeric=True)
        coalesce_columns(raw, "pedra_ano", ["pedra_2024", "pedra_2023", "pedra_23", "pedra_22"], numeric=False)
        coalesce_columns(raw, "ativo_inativo", ["ativo_inativo", "ativo_inativo_1"], numeric=False)

        for col in [
            "fase",
            "turma",
            "genero",
            "ano_ingresso",
            "pedra_20",
            "pedra_21",
            "pedra_22",
            "pedra_23",
            "inde_22",
            "inde_23",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "indicado",
            "atingiu_pv",
            "rec_psicologia",
        ]:
            if col not in raw.columns:
                raw[col] = np.nan

        phase_from_fase = raw["fase"].apply(extract_phase_code)
        phase_from_ideal = raw["fase_ideal"].apply(extract_phase_code)
        raw["fase_programa"] = phase_from_fase.where(phase_from_fase.notna(), phase_from_ideal)

        raw["idade"] = normalize_age_series(raw["idade"])

        numeric_columns = [
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]
        for n_col in numeric_columns:
            raw[n_col] = coerce_numeric_series(raw[n_col])

        score_columns = [
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]
        for s_col in score_columns:
            raw[s_col] = raw[s_col].clip(lower=0, upper=10)

        raw["genero"] = raw["genero"].apply(standardize_gender)
        raw["pedra_ano"] = raw["pedra_ano"].apply(standardize_stone)
        raw["pedra_22"] = raw["pedra_22"].apply(standardize_stone)
        raw["pedra_23"] = raw["pedra_23"].apply(standardize_stone)

        final_columns = [
            "ra",
            "ano_referencia",
            "nome_aluno",
            "data_nascimento",
            "idade",
            "genero",
            "ano_ingresso",
            "instituicao_ensino",
            "fase",
            "fase_ideal",
            "fase_programa",
            "turma",
            "pedra_20",
            "pedra_21",
            "pedra_22",
            "pedra_23",
            "pedra_ano",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "defasagem",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
            "indicado",
            "atingiu_pv",
            "ativo_inativo",
            "rec_psicologia",
        ]
        for col in final_columns:
            if col not in raw.columns:
                raw[col] = np.nan

        unified_frames.append(raw[final_columns].copy())

    if not unified_frames:
        raise ValueError("Nenhuma aba valida foi carregada do Excel.")

    df = pd.concat(unified_frames, ignore_index=True)
    df = df.drop_duplicates(subset=["ra", "ano_referencia"], keep="first")
    df = df.sort_values(["ra", "ano_referencia"]).reset_index(drop=True)

    cat_cols = [
        "nome_aluno",
        "data_nascimento",
        "genero",
        "instituicao_ensino",
        "fase",
        "fase_ideal",
        "fase_programa",
        "turma",
        "pedra_20",
        "pedra_21",
        "pedra_22",
        "pedra_23",
        "pedra_ano",
        "indicado",
        "atingiu_pv",
        "ativo_inativo",
        "rec_psicologia",
    ]
    for c_col in cat_cols:
        df[c_col] = clean_empty_strings(df[c_col])

    return df


def build_longitudinal_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Cria visao ano atual + ano seguinte por RA."""
    base = df.rename(columns={"ano_referencia": "ano_base"}).copy()
    future = df[["ra", "ano_referencia", "ian", "ida", "inde_ano", "ieg", "ips", "ipp", "ipv"]].copy()
    future = future.rename(
        columns={
            "ano_referencia": "ano_base",
            "ian": "ian_prox",
            "ida": "ida_prox",
            "inde_ano": "inde_prox",
            "ieg": "ieg_prox",
            "ips": "ips_prox",
            "ipp": "ipp_prox",
            "ipv": "ipv_prox",
        }
    )
    future["ano_base"] = future["ano_base"] - 1
    merged = base.merge(future, on=["ra", "ano_base"], how="left")
    merged["delta_ida_prox"] = merged["ida_prox"] - merged["ida"]
    merged["delta_inde_prox"] = merged["inde_prox"] - merged["inde_ano"]
    merged["delta_ian_prox"] = merged["ian_prox"] - merged["ian"]
    return merged


## 3) Carga da base consolidada

In [3]:
def load_consolidated_with_fallback() -> tuple[pd.DataFrame, str]:
    if DATA_PATH.exists():
        try:
            return load_and_prepare_data(DATA_PATH), "excel"
        except PermissionError as exc:
            print(f"Nao foi possivel ler o Excel por permissao: {exc}")
            print("Tentando fallback no CSV consolidado...")
        except Exception as exc:
            print(f"Falha na leitura do Excel: {exc}")
            print("Tentando fallback no CSV consolidado...")

    if CONSOLIDATED_FALLBACK_PATH.exists():
        fallback_df = pd.read_csv(CONSOLIDATED_FALLBACK_PATH)

        numeric_columns = [
            "ano_referencia",
            "idade",
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]
        for col in numeric_columns:
            if col in fallback_df.columns:
                if col == "idade":
                    fallback_df[col] = normalize_age_series(fallback_df[col])
                else:
                    fallback_df[col] = coerce_numeric_series(fallback_df[col])

        string_columns = [
            "ra",
            "nome_aluno",
            "data_nascimento",
            "genero",
            "instituicao_ensino",
            "fase",
            "fase_ideal",
            "fase_programa",
            "turma",
            "pedra_20",
            "pedra_21",
            "pedra_22",
            "pedra_23",
            "pedra_ano",
            "indicado",
            "atingiu_pv",
            "ativo_inativo",
            "rec_psicologia",
        ]
        for col in string_columns:
            if col in fallback_df.columns:
                fallback_df[col] = clean_empty_strings(fallback_df[col])

        fallback_df = fallback_df.drop_duplicates(subset=["ra", "ano_referencia"], keep="first")
        fallback_df = fallback_df.sort_values(["ra", "ano_referencia"]).reset_index(drop=True)
        return fallback_df, "csv"

    raise FileNotFoundError("Nao foi possivel carregar a base nem via Excel nem via CSV consolidado.")


df, fonte_dados = load_consolidated_with_fallback()
df_long = build_longitudinal_dataset(df)

if {"nota_matematica", "nota_portugues", "nota_ingles"}.issubset(df.columns):
    df["media_notas"] = df[["nota_matematica", "nota_portugues", "nota_ingles"]].mean(axis=1)
if {"iaa", "ieg", "ips", "ipp"}.issubset(df.columns):
    df["media_comportamental"] = df[["iaa", "ieg", "ips", "ipp"]].mean(axis=1)

print("Fonte utilizada:", fonte_dados)
print("Shape consolidado:", df.shape)
print("Anos na base:", sorted(df["ano_referencia"].dropna().unique().tolist()))
print("Alunos unicos no periodo:", df["ra"].nunique())

display(df.head(3))
display(
    df.groupby("ano_referencia", as_index=False)
    .agg(alunos=("ra", "nunique"), registros=("ra", "size"))
    .sort_values("ano_referencia")
)

df.to_csv(PROCESSED_DIR / "pede_consolidado_2022_2024.csv", index=False)
df_long.to_csv(PROCESSED_DIR / "pede_longitudinal_2022_2024.csv", index=False)
print("Arquivos salvos em:", PROCESSED_DIR.resolve())


Shape consolidado: (3027, 35)
Anos na base: [2022, 2023, 2024]
Alunos unicos no periodo: 1660


,ra,ano_referencia,nome_aluno,data_nascimento,idade,genero,ano_ingresso,instituicao_ensino,fase,fase_ideal,fase_programa,turma,pedra_20,pedra_21,pedra_22,pedra_23,pedra_ano,inde_22,inde_23,inde_ano,ian,ida,ieg,iaa,ips,ipp,ipv,defasagem,nota_matematica,nota_portugues,nota_ingles,indicado,atingiu_pv,ativo_inativo,rec_psicologia
0,RA-10,2022,Aluno-10,2004,18.0,Feminino,2021,Escola Pública,7,Fase 8 (Universitários),FASE 7,A,NaN,Ágata,Quartzo,NaN,Quartzo,5.784,NaN,5.7840,5.0,4.1,5.2,8.3,5.00,NaN,7.056,-1,3.3,2.6,6.4,Não,Não,NaN,Requer avaliação
1,RA-100,2022,Aluno-100,2009,13.0,Feminino,2019,Rede Decisão,4,Fase 3 (7º e 8º ano),FASE 4,A,Ametista,Topázio,Ametista,NaN,Ametista,7.618,NaN,7.6180,10.0,7.6,7.8,8.8,5.00,NaN,7.250,1,7.0,7.8,8.1,Não,Não,NaN,Não indicado
2,RA-1000,2023,Aluno-1000,4/20/2015,8.0,Feminino,2023,Pública,ALFA,ALFA (1° e 2° ano),ALFA,ALFA U - G2/G3,NaN,NaN,NaN,NaN,Ametista,NaN,NaN,7.9162,10.0,7.0,9.4,8.5,3.77,6.25,8.920,0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


,ano_referencia,alunos,registros
0,2022,859,859
1,2023,1013,1013
2,2024,1155,1155


Arquivos salvos em: C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\data\processed


## 4) Qualidade de dados

In [4]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).reset_index()
missing_pct.columns = ["coluna", "pct_faltante"]

display(missing_pct.head(20))

fig_missing = px.bar(
    missing_pct.head(20),
    x="pct_faltante",
    y="coluna",
    orientation="h",
    title="Top 20 colunas com maior percentual de faltantes",
    labels={"pct_faltante": "% faltante", "coluna": "Coluna"},
)
fig_missing.update_layout(yaxis={"categoryorder": "total ascending"})
fig_missing.show()

,coluna,pct_faltante
0,pedra_23,77.205154
1,inde_23,77.205154
2,pedra_20,75.189957
3,indicado,71.622068
4,atingiu_pv,71.622068
5,rec_psicologia,71.622068
6,pedra_21,65.047902
7,nota_ingles,63.990750
8,ativo_inativo,61.843409
9,inde_22,36.273538


## 5) Matriz de correlacao (pedido adicional)

In [ ]:
corr_features = [
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "inde_ano",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
]
corr_features = [c for c in corr_features if c in df.columns]

corr_matrix = df[corr_features].corr(method="spearman", min_periods=30)
display(corr_matrix.round(3))

fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Matriz de correlacao de Spearman - indicadores consolidados",
)
fig_corr.update_layout(coloraxis_colorbar={"title": "rho"})
fig_corr.show()

corr_inde_by_year = []
for year, part in df.groupby("ano_referencia"):
    required_cols = ["inde_ano", "ida", "ieg", "ips", "ipp", "ian", "ipv"]
    if not set(required_cols).issubset(part.columns):
        continue

    sub = part[required_cols].dropna()
    if len(sub) < 30:
        continue

    cm = sub.corr(method="spearman")
    corr_inde_by_year.append(
        {
            "ano_referencia": year,
            "corr_inde_ida": cm.loc["inde_ano", "ida"],
            "corr_inde_ieg": cm.loc["inde_ano", "ieg"],
            "corr_inde_ips": cm.loc["inde_ano", "ips"],
            "corr_inde_ipp": cm.loc["inde_ano", "ipp"],
            "corr_inde_ian": cm.loc["inde_ano", "ian"],
            "corr_inde_ipv": cm.loc["inde_ano", "ipv"],
            "n": len(sub),
        }
    )

corr_inde_by_year = pd.DataFrame(corr_inde_by_year).sort_values("ano_referencia")
display(corr_inde_by_year.round(3))


## 6) Q1 - IAN (Adequacao de Nivel): perfil de defasagem e evolucao temporal

In [5]:
ian_summary = (
    df.dropna(subset=["ian"])
    .groupby("ano_referencia", as_index=False)
    .agg(media_ian=("ian", "mean"), mediana_ian=("ian", "median"), alunos=("ra", "nunique"))
    .sort_values("ano_referencia")
)
display(ian_summary.round(3))

fig_ian_line = px.line(
    ian_summary,
    x="ano_referencia",
    y=["media_ian", "mediana_ian"],
    markers=True,
    title="Evolucao anual do IAN",
    labels={"value": "Score", "ano_referencia": "Ano"},
)
fig_ian_line.show()


def faixa_ian(value: float) -> str:
    if pd.isna(value):
        return np.nan
    if value <= 5:
        return "Defasagem alta (<=5)"
    if value <= 7:
        return "Atencao (5-7)"
    return "Adequado (>7)"


ian_mix = df[["ano_referencia", "ian"]].copy()
ian_mix["faixa_ian"] = ian_mix["ian"].apply(faixa_ian)
ian_mix = (
    ian_mix.dropna(subset=["faixa_ian"])
    .groupby(["ano_referencia", "faixa_ian"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)
ian_mix["percentual"] = ian_mix.groupby("ano_referencia")["quantidade"].transform(lambda s: 100 * s / s.sum())

fig_ian_mix = px.bar(
    ian_mix,
    x="ano_referencia",
    y="percentual",
    color="faixa_ian",
    barmode="stack",
    title="Composicao percentual de perfis de IAN por ano",
    labels={"percentual": "% de alunos", "ano_referencia": "Ano", "faixa_ian": "Faixa IAN"},
)
fig_ian_mix.show()

,ano_referencia,media_ian,mediana_ian,alunos
0,2022,6.426,5.0,859
1,2023,7.241,5.0,1013
2,2024,7.682,10.0,1155


## 7) Q2 - IDA (Desempenho Academico): melhora, estagnacao ou queda?

In [6]:
ida_summary = (
    df.dropna(subset=["ida"])
    .groupby("ano_referencia", as_index=False)
    .agg(media_ida=("ida", "mean"), mediana_ida=("ida", "median"), alunos=("ra", "nunique"))
    .sort_values("ano_referencia")
)
display(ida_summary.round(3))

fig_ida_line = px.line(
    ida_summary,
    x="ano_referencia",
    y=["media_ida", "mediana_ida"],
    markers=True,
    title="Evolucao anual do IDA",
    labels={"value": "Score", "ano_referencia": "Ano"},
)
fig_ida_line.show()

fig_ida_box = px.box(
    df.dropna(subset=["ida"]),
    x="ano_referencia",
    y="ida",
    color="ano_referencia",
    title="Distribuicao do IDA por ano",
    labels={"ano_referencia": "Ano", "ida": "IDA"},
)
fig_ida_box.update_layout(showlegend=False)
fig_ida_box.show()

,ano_referencia,media_ida,mediana_ida,alunos
0,2022,6.095,6.30,859
1,2023,6.663,6.80,937
2,2024,6.351,6.75,1055


## 8) Q3 - IEG (Engajamento) x IDA e IPV

In [7]:
corr_rows = []
for year, part in df.groupby("ano_referencia"):
    sub = part[["ieg", "ida", "ipv"]].dropna()
    if len(sub) < 10:
        continue
    corr_rows.append(
        {
            "ano_referencia": year,
            "corr_ieg_ida": sub["ieg"].corr(sub["ida"], method="spearman"),
            "corr_ieg_ipv": sub["ieg"].corr(sub["ipv"], method="spearman"),
            "corr_ida_ipv": sub["ida"].corr(sub["ipv"], method="spearman"),
            "n": len(sub),
        }
    )

corr_df = pd.DataFrame(corr_rows).sort_values("ano_referencia")
display(corr_df.round(3))

scatter_df = df.dropna(subset=["ieg", "ida", "ipv", "ano_referencia"]).copy()
fig_scatter = px.scatter(
    scatter_df,
    x="ieg",
    y="ida",
    color="ipv",
    facet_col="ano_referencia",
    color_continuous_scale="Viridis",
    opacity=0.70,
    hover_data=["ra", "fase_programa", "pedra_ano"],
    title="Relacao IEG x IDA (cor = IPV)",
    labels={"ieg": "IEG", "ida": "IDA", "ipv": "IPV"},
)
fig_scatter.show()

,ano_referencia,corr_ieg_ida,corr_ieg_ipv,corr_ida_ipv,n
0,2022,0.506,0.541,0.624,859
1,2023,0.446,0.491,0.551,937
2,2024,0.519,0.551,0.548,1054


## 9) Q4 - IAA (Autoavaliacao): coerencia com IDA e IEG

In [8]:
coherence = df.dropna(subset=["iaa", "ida", "ieg"]).copy()
coherence["media_objetiva"] = coherence[["ida", "ieg"]].mean(axis=1)
coherence["desvio_autoavaliacao"] = coherence["iaa"] - coherence["media_objetiva"]

coherence_summary = (
    coherence.groupby("ano_referencia", as_index=False)
    .agg(media_desvio=("desvio_autoavaliacao", "mean"), mediana_desvio=("desvio_autoavaliacao", "median"), alunos=("ra", "nunique"))
    .sort_values("ano_referencia")
)
display(coherence_summary.round(3))

fig_coherence = px.histogram(
    coherence,
    x="desvio_autoavaliacao",
    color="ano_referencia",
    barmode="overlay",
    nbins=40,
    opacity=0.60,
    title="Distribuicao do desvio entre IAA e media(IDA, IEG)",
    labels={"desvio_autoavaliacao": "IAA - media(IDA, IEG)"},
)
fig_coherence.show()

,ano_referencia,media_desvio,mediana_desvio,alunos
0,2022,1.279,1.35,859
1,2023,-0.765,0.45,937
2,2024,1.324,1.25,1054


## 10) Q5 e Q6 - IPS/IPP: padroes antecedentes e confirmacao com IAN

In [9]:
prior = df_long[df_long["ano_base"].isin([2022, 2023])].copy()
prior = prior.dropna(subset=["ida", "ida_prox", "ieg", "ieg_prox", "ips", "ipp", "ian", "ian_prox"])

prior["queda_ida_relevante"] = (prior["delta_ida_prox"] <= -1.0).astype(int)
prior["delta_ieg_prox"] = prior["ieg_prox"] - prior["ieg"]
prior["queda_ieg_relevante"] = (prior["delta_ieg_prox"] <= -1.0).astype(int)
prior["queda_ian_relevante"] = (prior["delta_ian_prox"] <= -1.0).astype(int)

risk_summary = []
for target_col, target_name in [
    ("queda_ida_relevante", "Queda IDA >= 1 ponto"),
    ("queda_ieg_relevante", "Queda IEG >= 1 ponto"),
    ("queda_ian_relevante", "Queda IAN >= 1 ponto"),
]:
    grouped = prior.groupby(target_col)[["ips", "ipp"]].mean()
    if {0, 1}.issubset(set(grouped.index)):
        risk_summary.append(
            {
                "evento": target_name,
                "ips_sem_evento": grouped.loc[0, "ips"],
                "ips_com_evento": grouped.loc[1, "ips"],
                "delta_ips": grouped.loc[1, "ips"] - grouped.loc[0, "ips"],
                "ipp_sem_evento": grouped.loc[0, "ipp"],
                "ipp_com_evento": grouped.loc[1, "ipp"],
                "delta_ipp": grouped.loc[1, "ipp"] - grouped.loc[0, "ipp"],
            }
        )

risk_summary_df = pd.DataFrame(risk_summary)
display(risk_summary_df.round(3))

prior["ips_baixo"] = (prior["ips"] <= prior["ips"].quantile(0.25)).astype(int)
prior["ipp_baixo"] = (prior["ipp"] <= prior["ipp"].quantile(0.25)).astype(int)

pattern_table = (
    prior.groupby(["ips_baixo", "ipp_baixo"], as_index=False)
    .agg(
        alunos=("ra", "nunique"),
        prob_queda_ida=("queda_ida_relevante", "mean"),
        prob_queda_ieg=("queda_ieg_relevante", "mean"),
        prob_queda_ian=("queda_ian_relevante", "mean"),
    )
    .sort_values(["ips_baixo", "ipp_baixo"])
)
for col in ["prob_queda_ida", "prob_queda_ieg", "prob_queda_ian"]:
    pattern_table[col] = pattern_table[col] * 100

display(pattern_table.round(2))

# Q6: IPP confirma/contradiz IAN
prior["perfil_ipp"] = np.where(prior["ipp"] <= 7, "IPP fragil (<=7)", "IPP adequado (>7)")
prior["perfil_ian"] = np.where(prior["ian"] <= 5, "IAN defasado (<=5)", "IAN adequado (>5)")

ipp_ian_matrix = pd.crosstab(prior["perfil_ipp"], prior["perfil_ian"], normalize="index") * 100
display(ipp_ian_matrix.round(2))

contradiction_rate = (
    ((prior["ipp"] > 7) & (prior["ian"] <= 5))
    | ((prior["ipp"] <= 7) & (prior["ian"] > 5))
).mean() * 100
print(f"Taxa de contradicao IPP x IAN: {contradiction_rate:.1f}%")

fig_pattern = px.bar(
    pattern_table.melt(
        id_vars=["ips_baixo", "ipp_baixo", "alunos"],
        value_vars=["prob_queda_ida", "prob_queda_ieg", "prob_queda_ian"],
        var_name="evento",
        value_name="probabilidade",
    ),
    x="evento",
    y="probabilidade",
    color="ipp_baixo",
    barmode="group",
    facet_col="ips_baixo",
    title="Probabilidade de quedas no ano seguinte por perfil psicossocial/psicopedagogico",
    labels={
        "evento": "Evento no ano seguinte",
        "probabilidade": "Probabilidade (%)",
        "ipp_baixo": "IPP baixo (1=Sim)",
        "ips_baixo": "IPS baixo (1=Sim)",
    },
)
fig_pattern.show()


,queda_ida_relevante,ips,ipp,ian,ieg,ida
0,0,5.190,7.552,7.402,8.845,6.642
1,1,5.098,7.721,6.748,8.894,7.151


ian_baixo,0,1
ipp_baixo,,
0,46.79,53.21
1,32.21,67.79


## 11) Q7 - IPV: comportamentos que mais influenciam o Ponto de Virada

In [10]:
ipv_features = ["ida", "ieg", "iaa", "ips", "ipp", "ian", "inde_ano"]
ipv_data = df.dropna(subset=ipv_features + ["ipv"]).copy()

if len(ipv_data) >= 100:
    rf_ipv = RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    rf_ipv.fit(ipv_data[ipv_features], ipv_data["ipv"])

    ipv_importance = pd.DataFrame({"feature": ipv_features, "importance": rf_ipv.feature_importances_}).sort_values(
        "importance", ascending=False
    )
    display(ipv_importance)

    fig_ipv_imp = px.bar(
        ipv_importance,
        x="importance",
        y="feature",
        orientation="h",
        title="Importancia relativa dos indicadores para explicar IPV",
        labels={"importance": "Importancia", "feature": "Indicador"},
    )
    fig_ipv_imp.update_layout(yaxis={"categoryorder": "total ascending"})
    fig_ipv_imp.show()
else:
    print("Amostra insuficiente para estimar importancia de IPV com robustez.")

,feature,importance
6,inde_ano,0.537874
4,ipp,0.232559
3,ips,0.069964
2,iaa,0.056906
1,ieg,0.042813
0,ida,0.033811
5,ian,0.026073


## 12) Q8 - Multidimensionalidade: combinacoes que explicam INDE

In [ ]:
multi_features = ["ida", "ieg", "ips", "ipp"]
multi_df = df.dropna(subset=multi_features + ["inde_ano"]).copy()

if len(multi_df) >= 150:
    X_train, X_test, y_train, y_test = train_test_split(
        multi_df[multi_features],
        multi_df["inde_ano"],
        test_size=0.25,
        random_state=42,
    )

    rf_inde = RandomForestRegressor(
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=6,
        random_state=42,
        n_jobs=-1,
    )
    rf_inde.fit(X_train, y_train)

    importance_inde = pd.DataFrame(
        {
            "indicador": multi_features,
            "importancia": rf_inde.feature_importances_,
        }
    ).sort_values("importancia", ascending=False)

    r2_train = rf_inde.score(X_train, y_train)
    r2_test = rf_inde.score(X_test, y_test)

    print(f"R2 treino (INDE): {r2_train:.3f}")
    print(f"R2 teste  (INDE): {r2_test:.3f}")
    display(importance_inde.round(4))

    fig_import_inde = px.bar(
        importance_inde,
        x="importancia",
        y="indicador",
        orientation="h",
        title="Importancia relativa (IDA + IEG + IPS + IPP) para explicar INDE",
        labels={"importancia": "Importancia", "indicador": "Indicador"},
    )
    fig_import_inde.update_layout(yaxis={"categoryorder": "total ascending"})
    fig_import_inde.show()
else:
    print("Amostra insuficiente para analise robusta da multidimensionalidade.")

# Perfis combinados (tercis) para leitura gerencial
profile_df = multi_df.copy()
for col in multi_features:
    profile_df[f"{col}_faixa"] = pd.qcut(profile_df[col], q=3, labels=["Baixo", "Medio", "Alto"], duplicates="drop")

profile_summary = (
    profile_df.groupby(["ida_faixa", "ieg_faixa", "ips_faixa", "ipp_faixa"], as_index=False)
    .agg(media_inde=("inde_ano", "mean"), alunos=("ra", "nunique"))
    .sort_values(["media_inde", "alunos"], ascending=[False, False])
)

profile_summary = profile_summary[profile_summary["alunos"] >= 15]
print("Top combinacoes de perfil (min. 15 alunos):")
display(profile_summary.head(10).round(3))


## 13) Q9 - Previsao de risco de defasagem (visao EDA + baseline de ML)

In [ ]:
risk_features = ["ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "inde_ano", "defasagem"]
risk_base = df_long[df_long["ano_base"].isin([2022, 2023])].copy()

risk_base["risco_defasagem_prox"] = (
    (risk_base["ian_prox"] <= 5)
    | (risk_base["delta_ian_prox"] <= -1)
).astype(int)

risk_model_df = risk_base.dropna(subset=risk_features + ["ian_prox"]).copy()

if len(risk_model_df) >= 200 and risk_model_df["risco_defasagem_prox"].nunique() == 2:
    X_train, X_test, y_train, y_test = train_test_split(
        risk_model_df[risk_features],
        risk_model_df["risco_defasagem_prox"],
        test_size=0.30,
        random_state=42,
        stratify=risk_model_df["risco_defasagem_prox"],
    )

    rf_risk = RandomForestClassifier(
        n_estimators=500,
        max_depth=7,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    rf_risk.fit(X_train, y_train)

    prob_test = rf_risk.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    print(f"ROC-AUC (teste): {roc_auc:.3f}")
    print(f"PR-AUC  (teste): {pr_auc:.3f}")

    risk_importance = pd.DataFrame(
        {
            "feature": risk_features,
            "importance": rf_risk.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    display(risk_importance.round(4))

    test_scored = X_test.copy()
    test_scored["prob_risco"] = prob_test
    test_scored["risco_real"] = y_test.values

    bins = pd.cut(
        test_scored["prob_risco"],
        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
        include_lowest=True,
    )
    calibration = (
        test_scored.groupby(bins, as_index=False)
        .agg(alunos=("risco_real", "size"), taxa_real=("risco_real", "mean"), prob_media=("prob_risco", "mean"))
    )
    calibration["taxa_real"] = calibration["taxa_real"] * 100
    calibration["prob_media"] = calibration["prob_media"] * 100
    display(calibration.round(2))
else:
    print("Amostra insuficiente para baseline de risco nesta visao EDA. Consulte o notebook de modelo preditivo.")


## 14) Q10 - Efetividade do programa por fases e coortes

In [11]:
program = df.dropna(subset=["ra", "ano_referencia", "pedra_ano"]).copy()
program = program[program["pedra_ano"].isin(["Quartzo", "Agata", "Ametista", "Topazio"])]

phase_metrics = (
    program.groupby(["pedra_ano", "ano_referencia"], as_index=False)
    .agg(
        alunos=("ra", "nunique"),
        media_inde=("inde_ano", "mean"),
        media_ida=("ida", "mean"),
        media_ieg=("ieg", "mean"),
        pct_ian_baixo=("ian", lambda s: (s <= 5).mean() * 100),
    )
    .sort_values(["pedra_ano", "ano_referencia"])
)
display(phase_metrics.round(3))

fig_phase_inde = px.line(
    phase_metrics,
    x="ano_referencia",
    y="media_inde",
    color="pedra_ano",
    markers=True,
    title="Efetividade por fase: evolucao de INDE",
    labels={"ano_referencia": "Ano", "media_inde": "INDE medio", "pedra_ano": "Fase"},
)
fig_phase_inde.show()

fig_phase_risk = px.line(
    phase_metrics,
    x="ano_referencia",
    y="pct_ian_baixo",
    color="pedra_ano",
    markers=True,
    title="Efetividade por fase: risco de defasagem (IAN <= 5)",
    labels={"ano_referencia": "Ano", "pct_ian_baixo": "% alunos com IAN <= 5", "pedra_ano": "Fase"},
)
fig_phase_risk.show()

# Visao de coorte por pedra inicial
cohort = program.dropna(subset=["inde_ano"]).copy()
first_record = (
    cohort.sort_values(["ra", "ano_referencia"])
    .groupby("ra", as_index=False)
    .first()[["ra", "pedra_ano"]]
    .rename(columns={"pedra_ano": "pedra_inicial"})
)
cohort = cohort.merge(first_record, on="ra", how="left")

cohort_evolution = (
    cohort.groupby(["pedra_inicial", "ano_referencia"], as_index=False)
    .agg(media_inde=("inde_ano", "mean"), alunos=("ra", "nunique"))
)

stone_order = ["Quartzo", "Agata", "Ametista", "Topazio"]
cohort_evolution["pedra_inicial"] = pd.Categorical(cohort_evolution["pedra_inicial"], categories=stone_order, ordered=True)
cohort_evolution = cohort_evolution.sort_values(["pedra_inicial", "ano_referencia"])
display(cohort_evolution.round(3))

fig_cohort = px.line(
    cohort_evolution,
    x="ano_referencia",
    y="media_inde",
    color="pedra_inicial",
    markers=True,
    title="Evolucao do INDE por coorte de pedra inicial",
    labels={"ano_referencia": "Ano", "media_inde": "INDE medio", "pedra_inicial": "Coorte inicial"},
)
fig_cohort.show()


,pedra_inicial,ano_referencia,media_inde,alunos
6,Quartzo,2022,5.239,131
7,Quartzo,2023,5.977,72
8,Quartzo,2024,5.875,77
0,Agata,2022,6.606,250
1,Agata,2023,6.662,244
2,Agata,2024,6.638,228
3,Ametista,2022,7.528,348
4,Ametista,2023,7.469,417
5,Ametista,2024,7.497,495
9,Topazio,2022,8.367,130


## 15) Q11 - Insights executivos, criatividade e checklist das perguntas

In [12]:
ian_risk = (
    df.dropna(subset=["ian"])
    .groupby("ano_referencia")["ian"]
    .apply(lambda s: (s <= 5).mean() * 100)
    .sort_index()
)
ida_means = df.dropna(subset=["ida"]).groupby("ano_referencia")["ida"].mean().sort_index()
delta_ida = ida_means.iloc[-1] - ida_means.iloc[0] if len(ida_means) >= 2 else np.nan

global_corr = df[["ieg", "ida", "ipp", "ian", "ips", "ipv", "inde_ano"]].corr(method="spearman")
corr_ieg_ida = global_corr.loc["ieg", "ida"]
corr_ipp_ian = global_corr.loc["ipp", "ian"]

cohort_delta = (
    cohort_evolution.groupby("pedra_inicial")
    .apply(lambda x: x.sort_values("ano_referencia")["media_inde"].iloc[-1] - x.sort_values("ano_referencia")["media_inde"].iloc[0])
    .dropna()
)

question_checklist = pd.DataFrame(
    [
        {"pergunta": "Q1 - IAN", "status": "Respondida", "evidencia": "Evolucao e composicao de faixas de defasagem por ano"},
        {"pergunta": "Q2 - IDA", "status": "Respondida", "evidencia": "Media/mediana e distribuicao anual"},
        {"pergunta": "Q3 - IEG x IDA/IPV", "status": "Respondida", "evidencia": "Correlacoes Spearman e grafico de dispersao"},
        {"pergunta": "Q4 - IAA coerente?", "status": "Respondida", "evidencia": "Desvio IAA vs media(IDA, IEG)"},
        {"pergunta": "Q5 - IPS antecede quedas?", "status": "Respondida", "evidencia": "Risco de queda futura de IDA/IEG/IAN por perfil IPS/IPP"},
        {"pergunta": "Q6 - IPP confirma IAN?", "status": "Respondida", "evidencia": "Matriz IPP x IAN e taxa de contradicao"},
        {"pergunta": "Q7 - Drivers do IPV", "status": "Respondida", "evidencia": "Importancia de variaveis com Random Forest"},
        {"pergunta": "Q8 - Multidimensionalidade", "status": "Respondida", "evidencia": "Modelo explicativo do INDE + perfis combinados"},
        {"pergunta": "Q9 - Previsao de risco", "status": "Respondida", "evidencia": "Baseline RF com probabilidade e metricas (ROC-AUC / PR-AUC)"},
        {"pergunta": "Q10 - Efetividade do programa", "status": "Respondida", "evidencia": "Tendencia por fase e por coorte"},
        {"pergunta": "Q11 - Insights e criatividade", "status": "Respondida", "evidencia": "Resumo executivo e recomendacoes acionaveis"},
    ]
)
display(question_checklist)

insight_text = f"""
### Principais achados

- **Risco de defasagem (IAN <= 5):**
  - 2022: {ian_risk.get(2022, np.nan):.1f}%
  - 2023: {ian_risk.get(2023, np.nan):.1f}%
  - 2024: {ian_risk.get(2024, np.nan):.1f}%
- **Tendencia de IDA (2022 -> 2024):** variacao media de **{delta_ida:.2f}** pontos.
- **Relacao IEG x IDA:** Spearman = **{corr_ieg_ida:.2f}**.
- **Relacao IPP x IAN:** Spearman = **{corr_ipp_ian:.2f}**.
- **Coortes por pedra (delta INDE no periodo):**
  {cohort_delta.round(2).to_string() if not cohort_delta.empty else 'Sem dados suficientes para coorte.'}

### Recomendacoes para a ONG

1. Priorizar trilhas de intervencao para alunos com **IAN <= 5** combinando reforco academico e suporte psicopedagogico.
2. Criar alertas preventivos quando houver queda simultanea de **IPS/IPP**, pois esse padrao antecede piora de IDA, IEG e IAN.
3. Usar **IEG como indicador de acao rapida**, por sua associacao com desempenho e ponto de virada.
4. Em alunos com desvio grande de autoavaliacao (IAA), aplicar devolutiva orientada para calibrar percepcao e metas.
5. Operacionalizar o baseline de risco em rotina mensal para priorizacao de acompanhamento individual.
"""
display(Markdown(insight_text))



### Principais achados

- **Risco de defasagem (IAN <= 5):**
  - 2022: 69.8%
  - 2023: 54.5%
  - 2024: 46.2%
- **Tendencia de IDA (2022 -> 2024):** variacao media de **0.26** pontos.
- **Relacao IEG x IDA:** Spearman = **0.50**.
- **Relacao IPP x IAN:** Spearman = **0.13**.
- **Coortes por pedra (delta INDE no periodo):**
  pedra_inicial
Quartzo     0.64
Agata       0.03
Ametista   -0.03
Topazio    -0.12

### Recomendacoes para a ONG

1. Priorizar trilhas de intervencao para alunos com **IAN <= 5** combinando reforco academico e suporte psicopedagogico.
2. Criar alertas preventivos quando houver queda simultanea de **IPS/IPP**, pois esse padrao antecede piora de IDA.
3. Usar **IEG como indicador de acao rapida**, por sua associacao com desempenho e ponto de virada.
4. Em alunos com desvio grande de autoavaliacao (IAA), aplicar devolutiva orientada para calibrar percepcao e metas.
5. Monitorar coortes por pedra com ritos de transicao claros (Quartzo -> Agata -> Ametista -> Topazio).
